# Does identifying core claims improve the argument map?

Each of Haiku, Sonnet and Opus builds an argument map from the whole document plus the atoms (`outputs/atomise_v01/opus/atoms.jsonl`), in two conditions:

| Condition | Extra input |
|---|---|
| `nocore` | none |
| `core` | the core propositions (union of Haiku and Opus `core` votes from `claim_core_test_v02`), flagged in the atom list |

Both conditions are told to rebuild only the core arguments, so not every atom is used. Statements are atom IDs; arguments follow the samples schema (conclusion, premises, conditions, qualifiers, concessions, risks, implications, alternatives), and an argument can be a member of another by ID.

Outputs: `outputs/argument_core_test/<condition>/<model>/` (`argument_map.json`, `raw*.txt`, `diagram.mmd`). Delete a run folder to re-run it.

In [ ]:
import json
import re
import sys
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from incolor.argumentation.backends.claude_agent import ClaudeAgentBackend

OUT = ROOT / "outputs" / "argument_core_test"
MODELS = ["haiku", "sonnet", "opus"]
CONDITIONS = ["nocore", "core"]
ROLES = ["premises", "conditions", "qualifiers", "concessions", "risks", "implications", "alternatives"]

DOC = re.sub(r"\A---.*?---\s*", "", (ROOT / "samples/BofA_Sept_FOMC_-_hike_or_bond_spike.txt").read_text(encoding="utf-8"), flags=re.S)
ATOMISE = ROOT / "outputs/atomise_v01/opus"
props = {
    m.group(1): m.group(2)
    for ln in (ATOMISE / "input.txt").read_text(encoding="utf-8").splitlines()
    if (m := re.match(r"\[(P\d+)\]\s+(.*)", ln))
}
atoms = {a["id"]: a for a in (json.loads(ln) for ln in (ATOMISE / "atoms.jsonl").read_text(encoding="utf-8").splitlines() if ln.strip())}
for a in atoms.values():
    a["pid"] = a["id"].split("-")[0]

# Core propositions: union of the Haiku and Opus votes
CORE_PROPS = sorted(
    {
        row["id"]
        for m in ("haiku", "opus")
        for row in json.loads((ROOT / f"outputs/claim_core_test_v02/{m}.json").read_text(encoding="utf-8"))["rows"]
        if row.get("core")
    }
)
CORE_ATOMS = {i for i, a in atoms.items() if a["pid"] in CORE_PROPS}
print(len(props), "propositions,", len(atoms), "atoms; core propositions:", CORE_PROPS, f"({len(CORE_ATOMS)} atoms)")

120 propositions, 269 atoms; core propositions: ['P005', 'P007', 'P009', 'P010'] (7 atoms)


In [2]:
SYSTEM = """You are a financial-research argument analyst. You get a sell-side research note in full, then its claims pre-extracted as atoms (IDs like P009-2, each with the extractor's modality tag), grouped under the sentence they came from.

Task: rebuild the CORE ARGUMENTS of the note as an argument map over the atoms. Do not use every atom: use only the atoms that do argumentative work for the author's main conclusions, and leave out the rest (background, data not needed as support, side scenarios, detail, restatements).

An argument has ONE conclusion (an atom ID) and the members that bear on it, each an atom ID or the ID of another argument (A###, meaning that argument's conclusion is itself a member, supported by its own members). Roles:
- premises: reasons or evidence the author gives for the conclusion
- conditions: what the conclusion depends on (if/unless/assuming)
- qualifiers: scope or strength limits on the conclusion
- concessions: points the author grants that cut against the conclusion
- risks: what could make the conclusion fail or hurt
- implications: what follows from the conclusion
- alternatives: the author's stated alternative outcome

Rules:
- Use the whole note to see how the author argues; link members only where the author states or clearly structures the link. Do not invent links.
- Use atom IDs exactly as given. Never rewrite atom text.
- Each atom appears at most once in the whole map (as one argument's conclusion or as one member of one argument). Where the note restates a point in several atoms, pick the clearest one.
- Each argument is a member of at most one other argument; no cycles. Top-level arguments are those nobody references.
- Every argument has at least one premise.
- Build hierarchy: if a premise has reasons of its own, make it the conclusion of its own argument and reference that argument's ID as the member.
- Summaries: one sentence, at most 30 words, using only claims of the member atoms.

Return ONE JSON object and nothing else (no code fences):
{"arguments": [{"id": "A001", "summary": "...", "conclusion": "P005-1", "premises": ["P012-1", "A002"], "conditions": [], "qualifiers": [], "concessions": [], "risks": [], "implications": [], "alternatives": []}]}
Number arguments A001, A002, ... in the order you list them."""

CORE_BLOCK = """
The author's headline (core) claims have been identified in a prior pass and are flagged CORE in the atom list. Build the map so that each top-level argument concludes one of the core claims (or an atom of a core proposition), or the core claims are reached through the hierarchy. Reasoning that does not bear on a core claim need not appear."""


def build_user(condition: str) -> str:
    blocks = []
    for pid, text in props.items():
        group = [a for a in atoms.values() if a["pid"] == pid]
        if not group:
            continue
        flag = " [CORE]" if condition == "core" and pid in CORE_PROPS else ""
        blocks.append(f"[{pid}]{flag} {text}\n" + "\n".join(f"  {a['id']} [{a['modality']}] {a['text']}" for a in group))
    extra = CORE_BLOCK if condition == "core" else ""
    return f"<document>\n{DOC}\n</document>\n\n<atoms>\n" + "\n".join(blocks) + f"\n</atoms>\n{extra}"


def parse(raw: str) -> dict:
    return json.loads(raw[raw.index("{") : raw.rindex("}") + 1])


def validate(m: dict) -> list[str]:
    """Structural checks on one argument map; returns error strings."""
    errors: list[str] = []
    args = {a.get("id"): a for a in m.get("arguments", [])}
    used: Counter = Counter()
    refs: Counter = Counter()
    for a in args.values():
        concl = a.get("conclusion")
        if concl not in atoms:
            errors.append(f"{a.get('id')}: conclusion {concl!r} is not an atom ID")
        else:
            used[concl] += 1
        members = [x for r in ROLES for x in a.get(r, [])]
        if not a.get("premises"):
            errors.append(f"{a.get('id')}: no premises")
        for x in members:
            if x in atoms:
                used[x] += 1
            elif x in args:
                refs[x] += 1
                if x == a.get("id"):
                    errors.append(f"{x}: references itself")
            else:
                errors.append(f"{a.get('id')}: unknown member {x!r}")
    errors += [f"atom {i} used {n} times" for i, n in used.items() if n > 1]
    errors += [f"argument {i} referenced {n} times" for i, n in refs.items() if n > 1]
    # cycle check on argument references
    state: dict[str, int] = {}

    def visit(i: str) -> None:
        if state.get(i) == 2 or i not in args:
            return
        if state.get(i) == 1:
            errors.append(f"cycle through {i}")
            return
        state[i] = 1
        for r in ROLES:
            for x in args[i].get(r, []):
                if x in args:
                    visit(x)
        state[i] = 2

    for i in args:
        visit(i)
    return errors


def run(job: tuple[str, str]) -> dict:
    condition, model = job
    folder = OUT / condition / model
    path = folder / "argument_map.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    folder.mkdir(parents=True, exist_ok=True)
    user = build_user(condition)
    backend = ClaudeAgentBackend(model=model)
    raws, errors, m = [], [], {}
    for attempt in range(2):
        prompt = user if attempt == 0 else user + f"\n\nYour previous answer had these errors; return a corrected full JSON object:\n- " + "\n- ".join(errors[:30]) + f"\n\nPrevious answer:\n{raws[-1]}"
        raws.append(backend.generate(SYSTEM, prompt, {}))
        (folder / f"raw{attempt + 1}.txt").write_text(raws[-1], encoding="utf-8")
        try:
            m = parse(raws[-1])
            errors = validate(m)
        except ValueError as exc:  # json.JSONDecodeError is a ValueError
            m, errors = {}, [f"unparseable JSON: {exc}"]
        if not errors:
            break
    result = {"map": m, "errors": errors, "attempts": len(raws), "usage": backend.usage}
    path.write_text(json.dumps(result, indent=1, ensure_ascii=False), encoding="utf-8")
    return result


jobs = [(c, m) for c in CONDITIONS for m in MODELS]
with ThreadPoolExecutor(len(jobs)) as pool:
    runs = dict(zip(jobs, pool.map(run, jobs)))

for (c, m), r in runs.items():
    cost = sum(u.get("cost_usd") or 0 for u in r["usage"])
    print(f"{c:7} {m:7} args={len(r['map'].get('arguments', []))} attempts={r['attempts']} errors={len(r['errors'])} ${cost:.3f}")

nocore  haiku   args=28 attempts=1 errors=0 $0.018
nocore  sonnet  args=7 attempts=2 errors=0 $0.209
nocore  opus    args=20 attempts=1 errors=0 $0.350
core    haiku   args=14 attempts=1 errors=0 $0.014
core    sonnet  args=4 attempts=1 errors=0 $0.097
core    opus    args=14 attempts=1 errors=0 $0.307


In [3]:
from itertools import combinations


def analyse(m: dict) -> dict:
    args = {a["id"]: a for a in m.get("arguments", []) if isinstance(a, dict) and "id" in a}
    referenced = {x for a in args.values() for r in ROLES for x in a.get(r, []) if x in args}
    roots = [i for i in args if i not in referenced]
    used = {a.get("conclusion") for a in args.values()} | {x for a in args.values() for r in ROLES for x in a.get(r, [])}
    used = {x for x in used if x in atoms}
    memo: dict[str, int] = {}

    def depth(i: str, seen=()) -> int:
        if i in seen:
            return 0
        if i not in memo:
            kids = [x for r in ROLES for x in args[i].get(r, []) if x in args]
            memo[i] = 1 + max((depth(k, (*seen, i)) for k in kids), default=0)
        return memo[i]

    root_atoms = {args[i].get("conclusion") for i in roots}
    concl_atoms = {a.get("conclusion") for a in args.values()}
    pid = lambda s: {atoms[x]["pid"] for x in s if x in atoms}
    return {
        "args": len(args),
        "roots": len(roots),
        "depth": max((depth(i) for i in args), default=0),
        "atoms_used": len(used),
        "props_used": pid(used),
        "root_props": pid(root_atoms),
        "concl_props": pid(concl_atoms),
        "roles": Counter(r for a in args.values() for r in ROLES for _ in a.get(r, [])),
    }


rows = []
info = {}
for (c, m), r in runs.items():
    x = analyse(r["map"])
    info[(c, m)] = x
    core_used = set(CORE_PROPS) & x["props_used"]
    rows.append(
        {
            "cond": c,
            "model": m,
            "errors": len(r["errors"]),
            "args": x["args"],
            "roots": x["roots"],
            "depth": x["depth"],
            "atoms_used": x["atoms_used"],
            "%atoms": round(100 * x["atoms_used"] / len(atoms)),
            "core_props_in_map": f"{len(core_used)}/{len(CORE_PROPS)}",
            "core_props_as_conclusion": f"{len(set(CORE_PROPS) & x['concl_props'])}/{len(CORE_PROPS)}",
            "core_props_as_root": f"{len(set(CORE_PROPS) & x['root_props'])}/{len(CORE_PROPS)}",
            "roots_that_are_core": f"{len(x['root_props'] & set(CORE_PROPS))}/{len(x['root_props'])}",
            "cost": round(sum(u.get('cost_usd') or 0 for u in r["usage"]), 3),
        }
    )
metrics = pd.DataFrame(rows)
print(metrics.to_string(index=False))


def jac(a: set, b: set) -> float:
    return len(a & b) / len(a | b) if a | b else 1.0


print("\nCross-model agreement (mean pairwise Jaccard)")
for c in CONDITIONS:
    pairs = list(combinations(MODELS, 2))
    for key in ("props_used", "root_props"):
        v = sum(jac(info[(c, a)][key], info[(c, b)][key]) for a, b in pairs) / len(pairs)
        print(f"  {c:7} {key:11}: {v:.2f}")

  cond  model  errors  args  roots  depth  atoms_used  %atoms core_props_in_map core_props_as_conclusion core_props_as_root roots_that_are_core  cost
nocore  haiku       0    28     10      4         107      40               4/4                      4/4                3/4                3/10 0.018
nocore sonnet       0     7      6      2          36      13               0/4                      0/4                0/4                 0/6 0.209
nocore   opus       0    20      2      5          92      34               3/4                      3/4                2/4                 2/2 0.350
  core  haiku       0    14      5      3          49      18               4/4                      4/4                4/4                 4/4 0.014
  core sonnet       0     4      3      2          22       8               3/4                      3/4                3/4                 3/3 0.097
  core   opus       0    14      3      4          85      32               4/4                     

In [ ]:
import yaml

from incolor.argumentation.diagram import ArgumentMapDiagram

# Atom-level types from claim_core_test (opus), mapped onto the standard theme's claim types
THEME_TYPE = {"FACT": "OBSERVATION", "VIEW": "VIEW", "FORECAST": "FORECAST", "PREDICTION": "PREDICTION", "ASSUMPTION": "SUPPOSITION"}
# Atomise v01 modalities mapped onto the theme's four
THEME_MODALITY = {
    "ASSERTED": "ASSERTED",
    "ATTRIBUTED": "ASSERTED",
    "EXPECTED": "HEDGED",
    "HEDGED": "HEDGED",
    "SPECULATIVE": "SPECULATIVE",
    "SUPPOSED": "HYPOTHETICAL",
}
atom_type = {
    r["id"]: THEME_TYPE.get(r["claim_type"].upper(), "VIEW")
    for r in json.loads((ROOT / "outputs/claim_core_test/opus.json").read_text(encoding="utf-8"))["rows"]
}


def to_theme_map(m: dict) -> dict:
    """Convert a run's map to the samples argument-map schema (atoms become statements)."""
    args = [a for a in m.get("arguments", []) if isinstance(a, dict) and "id" in a]
    used = {a["conclusion"] for a in args} | {x for a in args for r in ROLES for x in a.get(r, []) if x in atoms}
    statements = [
        {
            "id": i,
            "brief": a["text"],
            "statement": a["text"],
            "claim_type": atom_type.get(i, "VIEW"),
            "modality": THEME_MODALITY[a["modality"]],
            "evidence": [i],
        }
        for i, a in atoms.items()
        if i in used
    ]
    arguments = [
        {"id": a["id"], "summary": a.get("summary", ""), "conclusion": a["conclusion"], **{r: a.get(r, []) for r in ROLES}}
        for a in args
    ]
    return {"statements": statements, "arguments": arguments}


for (c, mdl), r in runs.items():
    themed = to_theme_map(r["map"])
    folder = OUT / c / mdl
    (folder / "argument_map.yaml").write_text(yaml.safe_dump(themed, sort_keys=False, allow_unicode=True), encoding="utf-8")
    ArgumentMapDiagram(themed).save(folder / "diagram.mmd")
print("wrote", len(runs), "themed diagram.mmd + argument_map.yaml files under", OUT)


def outline(c: str, mdl: str) -> str:
    m = runs[(c, mdl)]["map"]
    args = {a["id"]: a for a in m.get("arguments", []) if isinstance(a, dict) and "id" in a}
    referenced = {x for a in args.values() for r in ROLES for x in a.get(r, []) if x in args}
    out: list[str] = []

    def show(i: str, level: int, role: str = "") -> None:
        a = args[i]
        c_id = a["conclusion"]
        tag = "*" if c_id in CORE_ATOMS else " "
        out.append(f"{'    ' * level}{tag}{role + ': ' if role else ''}{c_id} {atoms.get(c_id, {}).get('text', '?')}")
        for r in ROLES:
            for x in a.get(r, []):
                if x in args:
                    show(x, level + 1, r)
                else:
                    t = "*" if x in CORE_ATOMS else " "
                    out.append(f"{'    ' * (level + 1)}{t}{r}: {x} {atoms.get(x, {}).get('text', '?')}")

    for i in args:
        if i not in referenced:
            show(i, 0)
            out.append("")
    return "\n".join(out)

wrote 6 themed diagram.mmd + argument_map.yaml files under c:\Users\angel.adeoye\dev\financial_research\outputs\argument_core_test


In [5]:
# * = atom of a core proposition
VIEW = [("nocore", "opus"), ("core", "opus")]
for c, mdl in VIEW:
    print(f"===== {c} / {mdl} =====")
    print(outline(c, mdl))

===== nocore / opus =====
*P009-1 Warsh delivers a hawkish hike.
    *premises: P005-1 The Fed hikes 25bp.
         premises: P013-1 Rates price the Sep hike with approximately 85% likelihood.
             premises: P014-1 The sharp shift in Fed pricing was catalyzed by the Warsh Jackson Hole (JH) speech.
             premises: P014-2 The sharp shift in Fed pricing was catalyzed by stronger labor data.
             premises: P014-3 The sharp shift in Fed pricing was catalyzed by stronger inflation data.
         premises: P030-1 Friday's CPI data all but lock in a rate hike this week.
         premises: P031-1 Chairman Warsh's Jackson Hole remarks have raised the cost of disappointing expectations significantly.
         premises: P031-2 Current market pricing has raised the cost of disappointing expectations significantly.
         premises: P032-3 Friday's twist flattening of the Treasury curve provides further justification for a hike.
         premises: P080-1 Rate pricing is firml